In [4]:
# Cell 1: Install and Import Libraries
!pip install -q scikit-learn pandas numpy nltk

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("Environment setup and imports completed successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Environment setup and imports completed successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# Cell 2: Dataset, Taxonomy, and Domain Concepts
# 1. Labelled Corpus (At least 40 queries across 5+ intent categories)
corpus_data = [
    # Intent 1: Symptoms
    ("What are the common symptoms of high blood sugar?", "Symptoms"),
    ("I am feeling excessively thirsty and fatigued all the time.", "Symptoms"),
    ("Is frequent urination a sign of diabetes?", "Symptoms"),
    ("Why am I experiencing sudden blurred vision?", "Symptoms"),
    ("Are slow-healing sores a symptom of diabetes?", "Symptoms"),
    ("Why am I losing weight unexpectedly?", "Symptoms"),
    ("Do I have tingling hands and feet?", "Symptoms"),
    ("Can extreme hunger be a sign of diabetes?", "Symptoms"),

    # Intent 2: Diet & Nutrition
    ("What foods should I avoid if I have diabetes?", "Diet"),
    ("Can I eat sweet fruits like mangoes and bananas?", "Diet"),
    ("What is a good meal plan for a diabetic patient?", "Diet"),
    ("Is brown rice better than white rice for blood sugar?", "Diet"),
    ("Are artificial sweeteners safe to use?", "Diet"),
    ("How many carbs should I eat per meal?", "Diet"),
    ("What are the best snacks for stabilizing blood sugar?", "Diet"),
    ("Can I drink fruit juice?", "Diet"),

    # Intent 3: Medication & Insulin
    ("How do I properly inject insulin?", "Medication"),
    ("What should I do if I miss a dose of my metformin?", "Medication"),
    ("Where is the best place on my body to inject insulin?", "Medication"),
    ("What are the side effects of diabetes pills?", "Medication"),
    ("How do I store my insulin vials?", "Medication"),
    ("Can insulin cause weight gain?", "Medication"),
    ("What is the difference between basal and bolus insulin?", "Medication"),
    ("Do I need to take pills and insulin together?", "Medication"),

    # Intent 4: Monitoring & Testing
    ("How many times a day should I check my blood glucose?", "Monitoring"),
    ("What is a normal fasting blood sugar level?", "Monitoring"),
    ("How do I use a glucometer correctly?", "Monitoring"),
    ("What is an HbA1c test and what should my target be?", "Monitoring"),
    ("When is the best time to check post-meal sugar?", "Monitoring"),
    ("What does continuous glucose monitoring (CGM) do?", "Monitoring"),
    ("Why is my morning blood sugar always high?", "Monitoring"),
    ("How often should I test my urine ketones?", "Monitoring"),

    # Intent 5: Lifestyle & Exercise
    ("What kind of exercises are safe for diabetics?", "Lifestyle"),
    ("Can regular walking lower my blood sugar?", "Lifestyle"),
    ("How does stress affect my glucose levels?", "Lifestyle"),
    ("Why is good sleep important for diabetes management?", "Lifestyle"),
    ("Should I check my blood sugar before working out?", "Lifestyle"),
    ("How can I manage diabetes while traveling?", "Lifestyle"),
    ("Can quitting smoking improve my insulin sensitivity?", "Lifestyle"),
    ("What precautions should I take before running?", "Lifestyle")
]

# 2. Hierarchical Taxonomy (4 major categories, 2 subcategories each)
taxonomy = {
    "Clinical Presentation": {
        "Subcategory_1": "Symptoms",
        "Subcategory_2": "Diagnosis"
    },
    "Nutritional Management": {
        "Subcategory_1": "Diet",
        "Subcategory_2": "Hydration"
    },
    "Medical Therapy": {
        "Subcategory_1": "Medication",
        "Subcategory_2": "Insulin Administration"
    },
    "Self-Care & Tracking": {
        "Subcategory_1": "Monitoring",
        "Subcategory_2": "Lifestyle"
    }
}

# 3. Domain Concepts (8+) & Relationships (10+)
domain_concepts = ["Patient", "Blood Glucose", "Insulin", "Carbohydrates", "HbA1c", "Pancreas", "Diet", "Glucometer"]
ontology_relationships = {
    "Patient": ["monitors Blood Glucose", "injects Insulin", "consumes Diet"],
    "Insulin": ["lowers Blood Glucose", "produced by Pancreas"],
    "Carbohydrates": ["raises Blood Glucose"],
    "HbA1c": ["measures average Blood Glucose"],
    "Glucometer": ["reads Blood Glucose"],
    "Diet": ["contains Carbohydrates"]
}

print(f"Total corpus size loaded: {len(corpus_data)} queries.")

Total corpus size loaded: 40 queries.


In [5]:
# Cell 3: NLP Preprocessing Function
# Ensure punkt_tab is downloaded
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase conversion
    text = text.lower()
    # 2. Tokenization & cleaning (remove punctuation/special characters)
    tokens = nltk.word_tokenize(text)
    tokens = [re.sub(r'[^a-zA-Z0-9]', '', token) for token in tokens]
    tokens = [token for token in tokens if token] # remove empty strings
    # 3. Stopword removal & 4. Lemmatization
    cleaned_tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]

    return " ".join(cleaned_tokens)

# Test preprocessing
sample_query = "What are the common symptoms of high blood sugar?!"
print("Original:", sample_query)
print("Preprocessed:", preprocess_text(sample_query))

Original: What are the common symptoms of high blood sugar?!
Preprocessed: common symptom high blood sugar


In [7]:
# Cell 4: Model Training, Testing, and Evaluation
df = pd.DataFrame(corpus_data, columns=['Query', 'Intent'])
df['Cleaned_Query'] = df['Query'].apply(preprocess_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Query'], df['Intent'], test_size=0.2, random_state=42, stratify=df['Intent']
)

# Build Pipeline: TF-IDF Vectorizer + Multinomial Naive Bayes Classifier
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

# Train the model
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=model_pipeline.classes_))

Accuracy Score: 37.50%

Classification Report:
              precision    recall  f1-score   support

        Diet       1.00      0.50      0.67         2
   Lifestyle       0.00      0.00      0.00         1
  Medication       0.00      0.00      0.00         2
  Monitoring       0.50      0.50      0.50         2
    Symptoms       1.00      1.00      1.00         1

    accuracy                           0.38         8
   macro avg       0.50      0.40      0.43         8
weighted avg       0.50      0.38      0.42         8

Confusion Matrix:
[[1 1 0 0 0]
 [0 0 0 1 0]
 [0 2 0 0 0]
 [0 1 0 1 0]
 [0 0 0 0 1]]


In [8]:
# Cell 5: Domain Knowledge Base and Rule-Based Logic
knowledge_base = {
    "Symptoms": "Common symptoms include excessive thirst, frequent urination, fatigue, and blurred vision. Please consult an endocrinologist for proper evaluation.",
    "Diet": "Focus on high-fiber foods, lean proteins, and complex carbohydrates. Limit sugary beverages, white bread, and processed sweets.",
    "Medication": "Take your prescribed medications (like Metformin) consistently with meals as directed by your physician. Never alter doses without consulting your doctor.",
    "Monitoring": "Check your fasting blood glucose regularly using a clean glucometer. Target ranges vary, but a common fasting goal is 80–130 mg/dL.",
    "Lifestyle": "Engage in at least 150 minutes of moderate aerobic exercise weekly, maintain healthy sleep hygiene, and manage daily stress."
}

def rule_engine(raw_query, predicted_intent, confidence_score):
    query_lower = raw_query.lower()

    # Rule 1: Emergency Escalation (Chest pain, severe hypoglycemia, ketoacidosis signs)
    emergency_keywords = ['chest pain', 'unconscious', 'severe chest', 'fainting', 'diabetic coma', 'suicide']
    if any(kw in query_lower for kw in emergency_keywords):
        return "[EMERGENCY ESCALATION]: Critical symptoms detected! Please call emergency services (911/112) or go to the nearest hospital immediately."

    # Rule 2: Unsupported Query Handling (Non-healthcare topics)
    supported_topics = ['sugar', 'diabetes', 'insulin', 'glucose', 'diet', 'food', 'symptom', 'medication', 'blood', 'pill', 'fatigue', 'carb', 'exercise', 'weight', 'thirsty', 'urin', 'vision', 'hba1c', 'glucometer', 'heal', 'sleep', 'stress']
    if not any(topic in query_lower for topic in supported_topics):
        return "[UNSUPPORTED QUERY]: I am a Diabetes Care Information Assistant. I can only answer queries related to diabetes care, symptoms, diet, monitoring, and medication."

    # Rule 3: Low-Confidence Handling (< 30% probability)
    if confidence_score < 0.30:
        return "[LOW CONFIDENCE]: I am not fully confident about your query. Could you please rephrase it or specify if it relates to symptoms, diet, medication, monitoring, or lifestyle?"

    # Rule 4: Urgent non-emergency advice
    if 'urgent' in query_lower or 'emergency' in query_lower:
        return "[URGENT NOTICE]: If you are experiencing sudden severe health changes, contact your healthcare provider immediately or visit urgent care."

    # Rule 5: Standard Knowledge Base Retrieval
    return knowledge_base.get(predicted_intent, "I recommend consulting your primary healthcare provider for personalized medical advice.")

In [ ]:
# Cell 6: Integrated Chatbot Function & Interactive Loop
def diabetes_assistant(user_input):
    # 1. Preprocess query
    cleaned = preprocess_text(user_input)

    # 2. Vectorize & Predict probabilities
    X_vec = model_pipeline.named_steps['tfidf'].transform([cleaned])
    probs = model_pipeline.named_steps['clf'].predict_proba(X_vec)[0]
    max_prob_idx = np.argmax(probs)
    predicted_intent = model_pipeline.classes_[max_prob_idx]
    confidence = probs[max_prob_idx]

    # 3. Apply Rule Engine & Knowledge Retrieval
    response = rule_engine(user_input, predicted_intent, confidence)
    return predicted_intent, confidence, response

# Interactive Chat Interface for Google Colab
print("--- Diabetes Care Information Assistant ---")
print("Type your health query below (type 'exit' to quit):\n")

while True:
    user_query = input("User: ")
    if user_query.lower() in ['exit', 'quit']:
        print("Assistant: Take care and stay healthy! Goodbye.")
        break
    if not user_query.strip():
        continue

    intent, conf, reply = diabetes_assistant(user_query)
    print(f" [Intent: {intent} | Confidence: {conf:.2f}]")
    print(f"Assistant: {reply}\n")

--- Diabetes Care Information Assistant ---
Type your health query below (type 'exit' to quit):

User: practicechatbit
 [Intent: Lifestyle | Confidence: 0.22]
Assistant: [UNSUPPORTED QUERY]: I am a Diabetes Care Information Assistant. I can only answer queries related to diabetes care, symptoms, diet, monitoring, and medication.

User: hi
 [Intent: Lifestyle | Confidence: 0.22]
Assistant: [UNSUPPORTED QUERY]: I am a Diabetes Care Information Assistant. I can only answer queries related to diabetes care, symptoms, diet, monitoring, and medication.

User: what should i avoid to eat in diabates
 [Intent: Diet | Confidence: 0.33]
Assistant: [UNSUPPORTED QUERY]: I am a Diabetes Care Information Assistant. I can only answer queries related to diabetes care, symptoms, diet, monitoring, and medication.

User: How often should I test my urine ketones?
 [Intent: Monitoring | Confidence: 0.34]
Assistant: Check your fasting blood glucose regularly using a clean glucometer. Target ranges vary, but 